# Blender One-Click Render — ChatGPT Work

Este cuaderno reemplaza las pruebas anteriores. Está diseñado para que el flujo normal sea:

1. Abrir en Google Colab.
2. Seleccionar **GPU T4** si Colab no la activó automáticamente.
3. Pulsar **Ejecutar todo**.
4. Autorizar Google Drive cuando Colab lo solicite.
5. Esperar. El render, logs, benchmark y archivos finales se guardan automáticamente en Drive.

Proyecto por defecto: `blender-neon-core-demo`.


In [ ]:
# @title Blender One-Click Render (solo ejecuta esta celda)
# Este cuaderno esta disenado para Google Colab. Usa por defecto el proyecto Neon Core.
# Para futuros proyectos solo cambia PROJECT_SLUG si hace falta.

PROJECT_SLUG = "blender-neon-core-demo"
REPO_URL = "https://github.com/izquierdodavid528-code/Chat-gpt-work-.git"
REPO_DIR = "/content/Chat-gpt-work-"
BLENDER_VERSION = "4.2.5"
BLENDER_DIR = f"/content/blender-{BLENDER_VERSION}-linux-x64"
BLENDER_BIN = f"{BLENDER_DIR}/blender"

import os
import sys
import json
import time
import shutil
import pathlib
import subprocess
import threading
from datetime import datetime

def sh(cmd, check=True, capture=False):
    print(f"\n$ {cmd}")
    if capture:
        p = subprocess.run(["bash","-lc",cmd], text=True, capture_output=True)
        if p.stdout:
            print(p.stdout)
        if p.stderr:
            print(p.stderr)
        if check and p.returncode != 0:
            raise RuntimeError(f"Comando fallo ({p.returncode}): {cmd}")
        return p
    return subprocess.run(["bash","-lc",cmd], check=check)

def section(title):
    print("\n" + "="*72)
    print(title)
    print("="*72)

section("1/8 - Verificando GPU de Colab")
gpu_info = sh(
    "nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader",
    check=False, capture=True
)
if gpu_info.returncode != 0 or not gpu_info.stdout.strip():
    raise RuntimeError(
        "No hay GPU NVIDIA activa. En Colab ve a Entorno de ejecucion > "
        "Cambiar tipo de entorno de ejecucion > GPU T4, guarda y vuelve a Ejecutar todo."
    )
print("GPU detectada correctamente.")

section("2/8 - Montando Google Drive")
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/Remotion Projects")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive OK:", DRIVE_ROOT)

section("3/8 - Preparando workspace")
if pathlib.Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)
sh(f'git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"')
os.chdir(REPO_DIR)
print("Workspace listo:", os.getcwd())

section("4/8 - Instalando Blender")
if not pathlib.Path(BLENDER_BIN).exists():
    url = f"https://download.blender.org/release/Blender4.2/blender-{BLENDER_VERSION}-linux-x64.tar.xz"
    sh("apt-get -qq update")
    sh("apt-get -qq install -y ffmpeg libegl1 libgl1 libx11-6 libxi6 libxrender1 libxfixes3 libxxf86vm1 libxkbcommon0")
    sh(f'wget -q --show-progress "{url}" -O /content/blender.tar.xz')
    sh("tar -xf /content/blender.tar.xz -C /content")
sh(f'"{BLENDER_BIN}" --version | head -n 3')

section("5/8 - Cargando proyecto")
project_dir = pathlib.Path(REPO_DIR) / "projects" / PROJECT_SLUG
config_path = project_dir / "project.config.json"
if not config_path.exists():
    raise FileNotFoundError(f"No existe: {config_path}")
cfg = json.load(open(config_path, encoding="utf-8"))
scene_script = project_dir / cfg.get("sceneScript", "scene.py")
if not scene_script.exists():
    raise FileNotFoundError(f"No existe: {scene_script}")
out_dir = project_dir / "out"
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps(cfg, indent=2, ensure_ascii=False))

section("6/8 - Renderizando con progreso")
progress_file = out_dir / "render-progress.json"
log_file = out_dir / "colab-render.log"
gpu_log = out_dir / "gpu-usage.csv"

env = os.environ.copy()
env["BLENDER_PROJECT_SLUG"] = PROJECT_SLUG
env["BLENDER_PROGRESS_FILE"] = str(progress_file)
# No forzamos software rendering. Queremos que Blender use el backend disponible en Colab.
env.pop("LIBGL_ALWAYS_SOFTWARE", None)
env.pop("MESA_LOADER_DRIVER_OVERRIDE", None)

cmd = [
    BLENDER_BIN,
    "-b",
    "--python",
    str(pathlib.Path(REPO_DIR) / "scripts" / "blender" / "render-with-progress.py"),
    "--",
    str(scene_script),
]
print("Comando:", " ".join(cmd))

stop_gpu_poll = threading.Event()

def poll_gpu():
    with open(gpu_log, "w", encoding="utf-8") as f:
        f.write("timestamp,gpu_util_percent,memory_used_mb\n")
        while not stop_gpu_poll.is_set():
            p = subprocess.run(
                ["bash","-lc",
                 "nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader,nounits"],
                text=True, capture_output=True
            )
            if p.returncode == 0 and p.stdout.strip():
                vals = p.stdout.strip().splitlines()[0]
                f.write(f"{datetime.utcnow().isoformat()}Z,{vals}\n")
                f.flush()
            stop_gpu_poll.wait(5)

poll_thread = threading.Thread(target=poll_gpu, daemon=True)
poll_thread.start()

start = time.time()
last_status = None
with open(log_file, "w", encoding="utf-8") as log:
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, text=True)
    while proc.poll() is None:
        if progress_file.exists():
            try:
                d = json.load(open(progress_file, encoding="utf-8"))
                status = (
                    d.get("status"),
                    d.get("percent"),
                    d.get("current_frame"),
                    d.get("frame_end"),
                    d.get("eta_seconds"),
                )
                if status != last_status:
                    pct = d.get("percent")
                    frame = d.get("current_frame")
                    end = d.get("frame_end")
                    eta = d.get("eta_seconds")
                    print(f"Progreso: {pct}% | frame {frame}/{end} | ETA {eta} s")
                    last_status = status
            except Exception:
                pass
        time.sleep(5)

elapsed = time.time() - start
stop_gpu_poll.set()
poll_thread.join(timeout=10)

print(f"\nRender finalizado en {elapsed/60:.2f} min ({elapsed:.1f} s)")
print("Return code:", proc.returncode)

if proc.returncode != 0:
    tail = "\n".join(open(log_file, errors="ignore").read().splitlines()[-100:])
    print("\nULTIMAS LINEAS DEL LOG:\n", tail)
    raise RuntimeError("Blender fallo. El log completo quedo guardado en out/colab-render.log")

section("7/8 - Normalizando video y verificando salidas")
expected_video = cfg.get("videoFile")
expected_video_path = pathlib.Path(REPO_DIR) / expected_video if expected_video else None

video_candidates = []
for ext in ("*.mp4", "*.mkv", "*.mov", "*.webm", "*.avi"):
    video_candidates.extend(out_dir.glob(ext))

# Si Blender produjo una extension/rango raro, normalizamos a MP4.
if expected_video_path and not expected_video_path.exists() and video_candidates:
    src = max(video_candidates, key=lambda p: p.stat().st_size)
    tmp_mp4 = out_dir / (expected_video_path.name + ".tmp.mp4")
    print("Normalizando video:", src.name, "->", expected_video_path.name)
    remux = subprocess.run([
        "ffmpeg","-y","-i",str(src),"-c","copy",str(tmp_mp4)
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if remux.returncode != 0:
        subprocess.run([
            "ffmpeg","-y","-i",str(src),
            "-c:v","libx264","-preset","veryfast","-crf","18",
            "-an",str(tmp_mp4)
        ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    tmp_mp4.replace(expected_video_path)

if expected_video_path and expected_video_path.exists():
    print("Video OK:", expected_video_path, expected_video_path.stat().st_size, "bytes")
else:
    print("AVISO: no se encontro el video esperado; revisa los archivos de out/.")

# Resumen de uso GPU
gpu_utils = []
if gpu_log.exists():
    for line in gpu_log.read_text(encoding="utf-8").splitlines()[1:]:
        try:
            parts = [x.strip() for x in line.split(",")]
            gpu_utils.append(float(parts[1]))
        except Exception:
            pass
if gpu_utils:
    print(f"GPU durante render: promedio {sum(gpu_utils)/len(gpu_utils):.1f}% | max {max(gpu_utils):.1f}%")
else:
    print("No se pudieron obtener muestras de uso GPU.")

summary = {
    "project": PROJECT_SLUG,
    "elapsed_seconds": round(elapsed, 1),
    "elapsed_minutes": round(elapsed/60, 2),
    "gpu": gpu_info.stdout.strip(),
    "gpu_util_avg_percent": round(sum(gpu_utils)/len(gpu_utils), 1) if gpu_utils else None,
    "gpu_util_max_percent": round(max(gpu_utils), 1) if gpu_utils else None,
    "blender_version": BLENDER_VERSION,
    "completed_at": datetime.utcnow().isoformat() + "Z",
}
(out_dir / "colab-benchmark.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

section("8/8 - Copiando todo a Google Drive")
drive_project = cfg.get("driveProjectDir", PROJECT_SLUG)
drive_out = DRIVE_ROOT / drive_project / "renders-colab"
drive_out.mkdir(parents=True, exist_ok=True)

for f in out_dir.iterdir():
    if f.is_file():
        shutil.copy2(f, drive_out / f.name)

print("\nLISTO.")
print("Destino:", drive_out)
print("Tiempo total:", summary["elapsed_minutes"], "min")
print("GPU media:", summary["gpu_util_avg_percent"], "%")
print("GPU maxima:", summary["gpu_util_max_percent"], "%")
print("\nArchivos copiados:")
for f in sorted(drive_out.iterdir()):
    if f.is_file():
        print(f" - {f.name}: {f.stat().st_size/1024/1024:.2f} MB")
